# Data on the Line · Setup test (5 minutes)

This notebook checks that the environment you'll use in the session is ready. You'll follow the same steps in the case.

1. Open the notebook in Colab with the link or QR code you were given. Colab downloads the data files from GitHub by itself.
2. Run all the cells in order with *Runtime → Run all*. At the end you should see **"All set."**

If you'd rather use Jupyter on your own computer, download the `Setup_Test` folder and open the notebook there.

If anything fails, send the error message to your instructor before the session.

In [ ]:
# on Colab the data files are downloaded from GitHub automatically; on your own computer this step is skipped
import os, subprocess, sys
if "google.colab" in sys.modules and not os.path.exists("data"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/burcu-koca-kd/data-on-the-line.git", "/content/data-on-the-line"], check=True)
    os.chdir("/content/data-on-the-line/en/1_Preparation/Setup_Test")
from pathlib import Path
print("Working folder:", Path.cwd())
print("data folder found ✓" if Path("data").exists() else "data folder NOT found: check the folder path.")

In [ ]:
def need(module, package=None):
    """Imports a library and installs it first if it is missing."""
    try:
        return __import__(module)
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q " + (package or module))
        return __import__(module)

need("highspy")                                            # backup solver, needed on some Macs
pulp = need("pulp", '"pulp>=2.7,<3"')
statsmodels, sklearn = need("statsmodels"), need("sklearn", "scikit-learn")
import pandas as pd, numpy as np, zipfile
print("pulp", pulp.__version__, "✓ | pandas", pd.__version__, "✓ | statsmodels", statsmodels.__version__, "✓ | scikit-learn", sklearn.__version__, "✓")

In [ ]:
result = {}
result["Reading a data file"] = len(pd.read_csv("data/sample.csv")) == 3

with zipfile.ZipFile("release_test.zip") as z:            # you will do exactly this at minutes 45 and 95
    z.extractall(".", pwd=b"test")
result["Unlocking a password-protected file"] = Path("release/test.csv").exists()

def solves(solver):
    """Checks whether the given solver can solve a tiny model; returns True if it can."""
    try:
        m = pulp.LpProblem("test", pulp.LpMinimize)
        x = pulp.LpVariable.dicts("x", range(3), cat="Binary")
        m += pulp.lpSum(x.values()); m += x[0] + x[1] >= 1; m += x[1] + x[2] >= 1
        m.solve(solver)
        return pulp.LpStatus[m.status] == "Optimal" and pulp.value(m.objective) == 1
    except Exception:
        return False

solver = "CBC" if solves(pulp.PULP_CBC_CMD(msg=False)) else "HiGHS" if solves(pulp.HiGHS(msg=False)) else None
result["Solving a small integer model" + (f" ({solver})" if solver else "")] = solver is not None

for name, ok in result.items():
    print(("✓ " if ok else "✗ ") + name)
if not solver:
    print("\nThe solver would not start. The easy fix is to use Colab. If you want to work on your own Mac, run `pip install highspy` in Terminal, restart the kernel and run this notebook again from the top.")
print("\nAll set." if all(result.values()) else "\nA step failed: send this message to your instructor.")